## Извлечение данных из архива

In [ ]:
import os
import shutil
import matplotlib.pyplot as plt
import cv2
import numpy as np
import random
import shutil
import json
import pandas as pd
import random
from skimage.color import label2rgb
from matplotlib.path import Path as pat
from pathlib import Path
from tqdm import tqdm
from utils.utils import read_config, get_image_in_folder
from utils.Image_processing import split_image_into_patch

In [ ]:
archives = [
    "Alabama Buildings Segmentation",
    "Buildings Instance Segmentation.v1-raw-images.coco-segmentation", 
    "California Satellite Images With Big Objects",
    "Mapping Challenge",
    "Massachusetts Buildings Dataset"
]

In [ ]:
for archive in archives:
    archive_path = f"datasets/{archive}.zip"  # или .tar, .tar.gz в зависимости от формата
    extract_path = f"datasets/{archive}"

    os.makedirs(extract_path, exist_ok=True)

    shutil.unpack_archive(archive_path, extract_path)
    print(f"Распакован: {archive}")

# Липецк

In [ ]:
path_to_lipetsk_image = Path("./datasets/NN_Igor/image")
paths_to_lipetsk_image = get_image_in_folder(path_to_lipetsk_image)
path_to_lipetsk_mask = Path("./datasets/NN_Igor/masks")
paths_to_lipetsk_mask = get_image_in_folder(path_to_lipetsk_mask)

In [ ]:
def conver_class(mask, unique_labels=None):

    converted_mask = np.zeros_like(mask, dtype=np.uint8)
    
    for class_id, brightness in enumerate(unique_labels):
        converted_mask[mask == brightness] = class_id
    
    return converted_mask

In [ ]:
images_visualization = 1
idx = 0
random.shuffle(paths_to_lipetsk_image)
# uniq_ien = [0, 126, 200, 81, 187]
# uniq_elec = [0, 99, 116, 122, 218]
uniq_lipetsk = [0, 109, 113, 125, 227]
# uniq_nizh_n = [0, 126, 200, 81, 187]
# uniq_nizh_e = [0, 126, 200, 81, 187]
# uniq_e = [0, 126, 200, 187, 81]
# colors = [
#     [0, 0, 0],       # 0: черный (фон)
#     [0, 0, 255],     # 83: синий
#     [255, 0, 0],     # 141: красный
#     [0, 255, 0],     # 134: зеленый
#     [255, 255, 0]    # 192: желтый
# ]
uniq = uniq_lipetsk
# Создаем цветовую палитру для разных классов
# [фон, класс1, класс2, класс3, класс4] - используем яркие цвета
colors_nizh_n = [
    [0, 0, 0],       # 0: черный (фон)
    [0, 0, 255],     # 83: синий
    [255, 0, 0],     # 141: красный
    [0, 255, 0],     # 134: зеленый
    [255, 255, 0]    # 192: желтый
]
colors = colors_nizh_n
for image_path in paths_to_lipetsk_image:
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")
    
    mask_path = image_path.parent.parent / 'masks' / image_path.name
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)  # Загружаем сразу в grayscale
    print("Уникальные лейблы:", np.unique(mask).astype(np.uint8))
    print("Количество лейблов:", len(np.unique(mask).astype(np.uint8)))
    mask = conver_class(mask, uniq)
    image_np = image
    mask_np = mask.astype(np.uint8)

    unique_labels = np.unique(mask_np)
    print(f"Имя файла: {image_path.name}")
    print("Уникальные лейблы:", unique_labels)
    print("Количество лейблов:", len(unique_labels))
    
    colored_mask = np.zeros_like(image_np)
    
    for i, label in enumerate(unique_labels):
        if label < len(colors):  
            color = colors[label % len(colors)] 
            colored_mask[mask_np == label] = color
    
    alpha = 0.5
    overlayed_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)
    
    segmented_regions = np.where(mask_np[:, :, None] != 0, overlayed_image, image_np)
    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)
    
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(image_np)
    plt.title('Original Image')
    plt.axis('off')

#     legend_elements = [plt.Rectangle((0, 0), 1, 1, facecolor=np.array(colors[i])/255) 
#                       for i in range(min(len(colors), len(unique_labels)))]

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np)
    plt.title('Mask (Grayscale)')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(segmented_regions)
    plt.title('Overlayed Image')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented Regions')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
# px_in_patch = [512, 1024]
path_to_save_lipetsk = Path("./datasets/prepared_dataset2/NN_Igor")
for i, image_path in enumerate(tqdm(paths_to_lipetsk_image)):
    px = 2048
    name = f"NN_Igor_{i}"

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    
    mask = conver_class(mask, uniq)

    split_image_into_patch(image,
                           mask,
                           px,
                           path_to_save_lipetsk,
                           name
                          )

In [ ]:
path_to_lipetsk_image = Path("./datasets/prepared_dataset2/NN_Igor/image")
paths_to_lipetsk_image = get_image_in_folder(path_to_lipetsk_image)
path_to_lipetsk_mask = Path("./datasets/prepared_dataset2/NN_Igor/masks")
paths_to_lipetsk_mask = get_image_in_folder(path_to_lipetsk_mask)

In [ ]:
print(f'image = {len(paths_to_lipetsk_image)}')
print(f'masks = {len(paths_to_lipetsk_mask)}')

In [ ]:
images_visualization = 15
idx = 0
random.shuffle(paths_to_lipetsk_image)

# [фон, класс1, класс2, класс3, класс4] - используем яркие цвета
# colors = [
#     [0, 0, 0],       # 0: черный (фон)
#     [0, 0, 255],     # 83: синий
#     [255, 0, 0],     # 141: красный
#     [0, 255, 0],     # 134: зеленый
#     [255, 255, 0]    # 192: желтый
# ]
colors = [
    [0, 0, 0],       # 0: черный (фон)
    [0, 0, 255],     # 83: синий
    [255, 0, 0],     # 141: красный
    [0, 255, 0],     # 134: зеленый
    [255, 255, 0]    # 192: желтый
]

for image_path in paths_to_lipetsk_image:
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")
    
    mask_path = image_path.parent.parent / 'masks' / image_path.name
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    image_np = image
    mask_np = mask.astype(np.uint8)

    unique_labels = np.unique(mask_np)
    print("Уникальные лейблы:", unique_labels)
    print("Количество лейблов:", len(unique_labels))
    
    colored_mask = np.zeros_like(image_np)
    
    for i, label in enumerate(unique_labels):
        if label < len(colors): 
            color = colors[label % len(colors)] 
            colored_mask[mask_np == label] = color
    
    alpha = 0.5
    overlayed_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)
    
    segmented_regions = np.where(mask_np[:, :, None] != 0, overlayed_image, image_np)
    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)
    
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(image_np)
    plt.title('Original Image')
    plt.axis('off')

    legend_elements = [plt.Rectangle((0, 0), 1, 1, facecolor=np.array(colors[i])/255) 
                      for i in range(min(len(colors), len(unique_labels)))]

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np)
    plt.title('Mask (Grayscale)')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(segmented_regions)
    plt.title('Overlayed Image')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented Regions')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
path_to_ien_image = Path("./datasets/prepared_dataset/Igor_Evgeniy_Nikolay/image")
paths_to_ien_image = get_image_in_folder(path_to_ien_image)
path_to_ien_mask = Path("./datasets/prepared_dataset/Igor_Evgeniy_Nikolay/masks")
paths_to_ien_mask = get_image_in_folder(path_to_ien_mask)

path_to_lipetsk_image = Path("./datasets/prepared_dataset/lipetsk/image")
paths_to_lipetsk_image = get_image_in_folder(path_to_lipetsk_image)
path_to_lipetsk_mask = Path("./datasets/prepared_dataset/lipetsk/masks")
paths_to_lipetsk_mask = get_image_in_folder(path_to_lipetsk_mask)

In [ ]:
print(len(paths_to_lipetsk_image))
print(len(paths_to_ien_image[:80]))

In [ ]:
def get_all_image_version(image_path):

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")
    mask_path = image_path.parent.parent / 'masks' / image_path.name
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

    image_np = image
    mask_np = mask.astype(np.uint8)

    unique_labels = np.unique(mask_np)
    print("Уникальные лейблы:", unique_labels)
    print("Количество лейблов:", len(unique_labels))
    
    colored_mask = np.zeros_like(image_np)
    
    for i, label in enumerate(unique_labels):
        if label < len(colors): 
            color = colors[label % len(colors)] 
            colored_mask[mask_np == label] = color
    
    alpha = 0.5
    overlayed_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)
    
    segmented_regions = np.where(mask_np[:, :, None] != 0, overlayed_image, image_np)
    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)


    return image_np, mask_np, segmented_regions, seg_house_im, unique_labels

In [ ]:
images_visualization = 15
idx = 0
random.shuffle(paths_to_lipetsk_image)
random.shuffle(paths_to_ien_image)
# Создаем цветовую палитру для разных классов
# [фон, класс1, класс2, класс3, класс4] - используем яркие цвета
# colors = [
#     [0, 0, 0],       # 0: черный (фон)
#     [0, 0, 255],     # 83: синий
#     [255, 0, 0],     # 141: красный
#     [0, 255, 0],     # 134: зеленый
#     [255, 255, 0]    # 192: желтый
# ]
colors = [
    [0, 0, 0],       # 0: черный (фон)
    [0, 0, 255],     # 83: синий
    [255, 0, 0],     # 141: красный
    [0, 255, 0],     # 134: зеленый
    [255, 255, 0]    # 192: желтый
]

for image_path_lipetsk, image_path_ien in zip(paths_to_lipetsk_image, paths_to_ien_image):
    
    image_np_lipetsk, mask_np_lipetsk, segmented_regions_lipetsk, seg_house_im_lipetsk, unique_labels_lipetsk = get_all_image_version(image_path_lipetsk)
    image_np_ien, mask_np_ien, segmented_regions_ien, seg_house_im_ien, unique_labels_ien = get_all_image_version(image_path_ien)
    
    plt.figure(figsize=(20, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(segmented_regions_lipetsk)
    plt.title('Overlayed Image Lipetsk')
    plt.axis('off')

    # legend_elements = [plt.Rectangle((0, 0), 1, 1, facecolor=np.array(colors[i])/255) 
    #                   for i in range(min(len(colors), len(unique_labels_lipetsk)))]

    plt.subplot(1, 4, 2)
    plt.imshow(seg_house_im_lipetsk)
    plt.title('Segmented Regions Lipetsk')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(segmented_regions_ien)
    plt.title('Overlayed Image Ien')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im_ien)
    plt.title('Segmented Regions Ien')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    if idx == images_visualization - 1:
        break
    idx += 1

## Проверка типов данных и их визуаллизация

### Alabama Buildings Segmentation

In [ ]:
path_to_alabama_image = Path("./datasets/Alabama Buildings Segmentation/alabama/image")
paths_to_alabama_image = get_image_in_folder(path_to_alabama_image)
path_to_alabama_mask = Path("./datasets/Alabama Buildings Segmentation/alabama/mask")
paths_to_alabama_mask = get_image_in_folder(path_to_alabama_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_alabama_image)
for image_path in paths_to_alabama_image:
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'mask'/image_path.name
    mask = cv2.imread(mask_path)

    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.subplot(1, 2, 2)
    plt.imshow(mask)
    if idx == images_visualization:
        break
    idx += 1 

Маски у домов инвертированные, нужно инвертировать их пересохранить

In [ ]:
def invert_and_save_masks(mask_paths):

    for mask_path in tqdm(mask_paths, desc="Инвертирование масок"):
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        inverted_mask = cv2.bitwise_not(mask)
        
        cv2.imwrite(str(mask_path), inverted_mask)

In [ ]:
invert_and_save_masks(paths_to_alabama_mask)

In [ ]:
path_to_alabama_image = Path("./datasets/Alabama Buildings Segmentation/alabama/image")
paths_to_alabama_image = get_image_in_folder(path_to_alabama_image)
path_to_alabama_mask = Path("./datasets/Alabama Buildings Segmentation/alabama/mask")
paths_to_alabama_mask = get_image_in_folder(path_to_alabama_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_alabama_image)
for image_path in paths_to_alabama_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'mask'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  # Красный цвет

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
px_in_patch = 256
path_to_save_alabama = Path("./datasets/prepared_dataset/alabama")
name = "alabama"
for i, image_path in enumerate(tqdm(paths_to_alabama_image)):
    name = f"alabama_{i}"
#     if len(list(Path(path_to_save_alabama/"image").glob(f"{name}0.png"))) == 0:
#         continue
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'mask'/image_path.name
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    split_image_into_patch(image,
                           mask,
                           px_in_patch,
                           path_to_save_alabama,
                           name
                          )

In [ ]:
path_to_alabama_image = Path("./datasets/prepared_dataset/alabama/image")
paths_to_alabama_image = get_image_in_folder(path_to_alabama_image)
path_to_alabama_mask = Path("./datasets/prepared_dataset/alabama/masks")
paths_to_alabama_mask = get_image_in_folder(path_to_alabama_mask)

In [ ]:
print(f'image = {len(paths_to_alabama_image)}')
print(f'masks = {len(paths_to_alabama_mask)}')

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_alabama_image)
for image_path in paths_to_alabama_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  # Красный цвет

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

### Buildings Instance Segmentation.v1-raw-images.coco-segmentation

In [ ]:
path_to_Buildings_Instance_image = Path("./datasets/Buildings Instance Segmentation.v1-raw-images.coco-segmentation/alabama/image")
paths_to_Buildings_Instance_image = get_image_in_folder(path_to_Buildings_Instance_image)
path_to_Buildings_Instance_mask = Path("./datasets/Buildings Instance Segmentation.v1-raw-images.coco-segmentation/alabama/mask")
paths_to_Buildings_Instance_mask = get_image_in_folder(path_to_Buildings_Instance_mask)

### California Satellite Images With Big Objects

In [ ]:
path_to_california_image = Path("./datasets/California Satellite Images With Big Objects/california/image")
paths_to_california_image = get_image_in_folder(path_to_california_image)
path_to_california_mask = Path("./datasets/California Satellite Images With Big Objects/california/mask")
paths_to_california_mask = get_image_in_folder(path_to_california_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_california_image)
for image_path in paths_to_california_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'mask'/image_path.name
    mask = cv2.imread(mask_path)

    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.subplot(1, 2, 2)
    plt.imshow(mask)
    if idx == images_visualization:
        break
    idx += 1 

In [ ]:
invert_and_save_masks(paths_to_california_mask)

In [ ]:
path_to_california_image = Path("./datasets/California Satellite Images With Big Objects/california/image")
paths_to_california_image = get_image_in_folder(path_to_california_image)
path_to_california_mask = Path("./datasets/California Satellite Images With Big Objects/california/mask")
paths_to_california_mask = get_image_in_folder(path_to_california_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_california_image)
for image_path in paths_to_california_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'mask'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  # Красный цвет

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
path_to_california_image = Path("./datasets/California Satellite Images With Big Objects/california/image")
paths_to_california_image = get_image_in_folder(path_to_california_image)
path_to_california_mask = Path("./datasets/California Satellite Images With Big Objects/california/mask")
paths_to_california_mask = get_image_in_folder(path_to_california_mask)

In [ ]:
px_in_patch = 256
path_to_save_california = Path("./datasets/prepared_dataset/california")
for i, image_path in enumerate(tqdm(paths_to_california_image)):
    name = f"california_{i}"

    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'mask'/image_path.name
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    split_image_into_patch(image,
                           mask,
                           px_in_patch,
                           path_to_save_california,
                           name
                          )

In [ ]:
path_to_california_image = Path("./datasets/prepared_dataset/california/image")
paths_to_california_image = get_image_in_folder(path_to_california_image)
path_to_california_mask = Path("./datasets/prepared_dataset/california/masks")
paths_to_california_mask = get_image_in_folder(path_to_california_mask)

In [ ]:
print(f'image = {len(paths_to_california_image)}')
print(f'masks = {len(paths_to_california_mask)}')

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_california_image)
for image_path in paths_to_california_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  # Красный цвет

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')a

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

### Mapping Challenge

In [ ]:
path_to_mapping_image = Path("./datasets/Mapping Challenge/train/images")
paths_to_mapping_image = get_image_in_folder(path_to_mapping_image)
path_to_mapping_annotation_mask = Path("./datasets/Mapping Challenge/annotation.json")
path_to_mapping_masks = Path("./datasets/Mapping Challenge/masks")

In [ ]:
with open(path_to_mapping_annotation_mask, 'r') as f:
    annot_data = json.load(f)

In [ ]:
image_df = pd.DataFrame(annot_data['images'])
image_df.sample(3)

In [ ]:
annot_df = pd.DataFrame(annot_data['annotations'])
annot_df.sample(3)

In [ ]:
annot_df_with_filename = annot_df.merge(
    image_df[['id', 'file_name']], 
    left_on='image_id', 
    right_on='id',
    how='left'
)

annot_df_with_filename = annot_df_with_filename.rename(columns={
    'id_x': 'id', 
    'id_y': 'image_df_id' 
})

annot_df_with_filename = annot_df_with_filename.drop('image_df_id', axis=1)

In [ ]:
full_df = pd.merge(annot_df, image_df, how='left', left_on = 'image_id', right_on='id').dropna()
print(image_df.shape[0], '+', annot_df.shape[0], '->', full_df.shape[0])
full_df.sample(2)

In [ ]:
def rows_to_segmentation(in_img, in_df):
    xx, yy = np.meshgrid(range(in_img.shape[0]), 
                range(in_img.shape[1]),
               indexing='ij')
    out_img = np.zeros(in_img.shape[:2])
    for _, c_row in in_df.iterrows():
        xy_vec = np.array(c_row['segmentation']).reshape((-1, 2))
        xy_path = pat(xy_vec)
        out_img += xy_path.contains_points(np.stack([yy.ravel(), 
                                                     xx.ravel()], -1)).reshape(out_img.shape)
    return out_img

In [ ]:
binary_mask = (out_img * 255).astype(np.uint8)

In [ ]:
images_visualization = 10
idx = 0
for (c_id, c_df) in tqdm(full_df.groupby('image_id')):

    img_data = cv2.imread(os.path.join(path_to_mapping_image, c_df['file_name'].values[0]))
    img_data = cv2.cvtColor(img_data, cv2.COLOR_BGR2RGB)
    out_img = rows_to_segmentation(img_data, c_df)
    binary_mask = (out_img * 255).astype(np.uint8)
    rgba_img = np.concatenate([img_data,
                               np.clip(np.expand_dims(127*out_img+127, -1), 0, 255).astype(np.uint8)
                              ], -1)

    cv2.imwrite(path_to_mapping_masks / c_df['file_name'].values[0], binary_mask)


In [ ]:
path_to_mapping_image = Path("./datasets/Mapping Challenge/train")
paths_to_mapping_image = get_image_in_folder(path_to_mapping_image)
path_to_mapping_masks = Path("./datasets/Mapping Challenge/masks")
paths_to_mapping_masks = get_image_in_folder(path_to_mapping_masks)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_mapping_image)
for image_path in paths_to_mapping_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)

    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255 

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')
    
    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
def move_images(image_paths, destination_folder):

    destination = Path(destination_folder)
    destination.mkdir(parents=True, exist_ok=True)
    
    moved_count = 0
    for image_path in tqdm(image_paths):
        source = Path(image_path)

        destination_file = destination / source.name

        counter = 1
        original_dest = destination_file
        while destination_file.exists():
            stem = original_dest.stem
            suffix = original_dest.suffix
            destination_file = original_dest.parent / f"{stem}_{counter}{suffix}"
            counter += 1

        shutil.move(str(source), str(destination_file))
        moved_count += 1

In [ ]:
move_images(paths_to_mapping_image, "./datasets/prepared_dataset/lost_city/image")

In [ ]:
move_images(paths_to_mapping_masks, "./datasets/prepared_dataset/lost_city/masks")

In [ ]:
path_to_mapping_image = Path("./datasets/prepared_dataset/lost_city/image")
paths_to_mapping_image = get_image_in_folder(path_to_mapping_image)
path_to_mapping_masks = Path("./datasets/prepared_dataset/lost_city/masks")
paths_to_mapping_masks = get_image_in_folder(path_to_mapping_masks)

In [ ]:
print(f'image = {len(paths_to_mapping_image)}')
print(f'masks = {len(paths_to_mapping_masks)}')

### Massachusetts Buildings Dataset

In [ ]:
def reorganize_folders(base_path):

    base_path = Path(base_path)
    
    images_dir = base_path / "images"
    masks_dir = base_path / "masks"
    
    images_dir.mkdir(exist_ok=True)
    masks_dir.mkdir(exist_ok=True)
    
    source_folders = {
        'train': images_dir,
        'test': images_dir, 
        'val': images_dir,
        'train_labels': masks_dir,
        'test_labels': masks_dir,
        'val_labels': masks_dir
    }
    
    for source_folder_name, target_dir in source_folders.items():
        source_folder = base_path / source_folder_name

        for file_path in source_folder.iterdir():
            if file_path.is_file():
                target_file = target_dir / file_path.name
                
                counter = 1
                original_target = target_file
                while target_file.exists():
                    stem = original_target.stem
                    suffix = original_target.suffix
                    target_file = original_target.parent / f"{stem}_{counter}{suffix}"
                    counter += 1
                
                shutil.move(str(file_path), str(target_file))
        
        try:
            source_folder.rmdir()
        except OSError:
            print(f"Папка {source_folder} не пуста, оставлена")

In [ ]:
base_path = "./datasets/Massachusetts Buildings Dataset/png"
reorganize_folders(base_path)

In [ ]:
path_to_massachusetts_image = Path("./datasets/Massachusetts Buildings Dataset/png/images")
paths_to_massachusetts_image = get_image_in_folder(path_to_massachusetts_image)
path_to_massachusetts_mask = Path("./datasets/Massachusetts Buildings Dataset/png/masks")
paths_to_massachusetts_mask = get_image_in_folder(path_to_massachusetts_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_massachusetts_image)
for image_path in paths_to_massachusetts_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  # Красный цвет

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)
    print(f'{image_path}')
    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
path_to_massachusetts_image = Path("./datasets/Massachusetts Buildings Dataset/png/images")
paths_to_massachusetts_image = get_image_in_folder(path_to_massachusetts_image)
path_to_massachusetts_mask = Path("./datasets/Massachusetts Buildings Dataset/png/masks")
paths_to_massachusetts_mask = get_image_in_folder(path_to_massachusetts_mask)

In [ ]:
px_in_patch = 256
path_to_save_massachusetts = Path("./datasets/prepared_dataset/massachusetts")
for i, image_path in enumerate(tqdm(paths_to_massachusetts_image)):
    name = f"massachusetts_{i}"

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    split_image_into_patch(image,
                           mask,
                           px_in_patch,
                           path_to_save_massachusetts,
                           name
                          )

In [ ]:
path_to_massachusetts_image = Path("./datasets/prepared_dataset/massachusetts/image")
paths_to_massachusetts_image = get_image_in_folder(path_to_massachusetts_image)
path_to_massachusetts_mask = Path("./datasets/prepared_dataset/massachusetts/masks")
paths_to_massachusetts_mask = get_image_in_folder(path_to_massachusetts_mask)

In [ ]:
print(f'image = {len(paths_to_massachusetts_image)}')
print(f'masks = {len(paths_to_massachusetts_mask)}')

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_massachusetts_image)
for image_path in paths_to_massachusetts_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  # Красный цвет

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

# All dataset size

In [ ]:
path_to_all_image = Path("./datasets/prepared_dataset/")
paths_to_all_image = get_image_in_folder(path_to_all_image)

In [ ]:
base_path = Path("./datasets/prepared_dataset")

image_files = []
for image_path in base_path.glob("*/image/*"):
    if image_path.is_file() and image_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
        image_files.append(image_path)

print(f"Найдено {len(image_files)} изображений")

In [ ]:
base_path = Path("./datasets/prepared_dataset")

image_files = []
for image_path in base_path.glob("*/masks/*"):
    if image_path.is_file() and image_path.suffix.lower() in ['.png', '.jpg', '.jpeg']:
        image_files.append(image_path)

print(f"Найдено {len(image_files)} масок")

# Зархивировать датасеты

In [ ]:
def create_archive(folder_path: str, output_path: str = None) -> str:

    folder_path = Path(folder_path)
    
    if not folder_path.exists():
        raise ValueError(f"Папка {folder_path} не существует")
    
    if not folder_path.is_dir():
        raise ValueError(f"{folder_path} не является папкой")
    
    if output_path is None:
        output_path = folder_path.parent / f"{folder_path.name}_archive"
    
    output_path = Path(output_path)
    
    archive_path = shutil.make_archive(
        base_name=str(output_path),
        format='zip',
        root_dir=folder_path.parent,
        base_dir=folder_path.name
    )
    
    return archive_path

In [ ]:
archive = create_archive(folder_path = "./datasets/prepared_dataset", output_path = "./datasets")

# Датасеты БПЛА или Ажрофотосъемки

In [ ]:
archives = [
    "aerial-imagery-for-roof-segmentation",
    "LandCover", 
    "Toronto"
]

In [ ]:
for archive in archives:
    archive_path = f"datasets/{archive}.zip" 
    extract_path = f"datasets/{archive}"

    os.makedirs(extract_path, exist_ok=True)

    shutil.unpack_archive(archive_path, extract_path)
    print(f"Распакован: {archive}")

## Aerial-imagery-for-roof-segmentation

In [ ]:
path_to_aerials_image_1 = Path("./datasets/aerial-imagery-for-roof-segmentation/test/image")
paths_to_aerials_image_1 = get_image_in_folder(path_to_aerials_image_1)
path_to_aerials_image_2 = Path("./datasets/aerial-imagery-for-roof-segmentation/train/image")
paths_to_aerials_image_2 = get_image_in_folder(path_to_aerials_image_2)
path_to_aerials_image_3 = Path("./datasets/aerial-imagery-for-roof-segmentation/val/image")
paths_to_aerials_image_3 = get_image_in_folder(path_to_aerials_image_3)


path_to_aerials_masks_1 = Path("./datasets/aerial-imagery-for-roof-segmentation/test/label")
paths_to_aerials_masks_1 = get_image_in_folder(path_to_aerials_masks_1)
path_to_aerials_masks_2 = Path("./datasets/aerial-imagery-for-roof-segmentation/train/label")
paths_to_aerials_masks_2 = get_image_in_folder(path_to_aerials_masks_2)
path_to_aerials_masks_3 = Path("./datasets/aerial-imagery-for-roof-segmentation/val/label")
paths_to_aerials_masks_3 = get_image_in_folder(path_to_aerials_masks_3)

paths_to_aerials_image = paths_to_aerials_image_1 + paths_to_aerials_image_2 + paths_to_aerials_image_3
paths_to_aerials_masks = paths_to_aerials_masks_1 + paths_to_aerials_masks_2 + paths_to_aerials_masks_3

In [ ]:
images_visualization = 1
idx = 0
random.shuffle(paths_to_aerials_image)
for image_path in paths_to_aerials_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")
    mask_path = image_path.parent.parent/'label'/image_path.name
    mask = cv2.imread(mask_path)

    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255 

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')
    
    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
px_in_patch = 512
path_to_save_aerial = Path("./datasets/prepared_dataset/aerial-imagery-for-roof-segmentation")
for i, image_path in enumerate(tqdm(paths_to_aerials_image)):
    name = f"aerial_{i}"
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'label'/image_path.name
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    split_image_into_patch(image,
                           mask,
                           px_in_patch,
                           path_to_save_aerial,
                           name
                          )

In [ ]:
path_to_aerial_image = Path("./datasets/prepared_dataset/aerial/image")
paths_to_aerial_image = get_image_in_folder(path_to_aerial_image)
path_to_aerial_mask = Path("./datasets/prepared_dataset/aerial/masks")
paths_to_aerial_mask = get_image_in_folder(path_to_aerial_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_aerial_image)
for image_path in paths_to_aerial_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255  

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

## Potsdam

In [ ]:
path_to_Potsdam_image = Path("./datasets/Potsdam/images")
paths_to_Potsdam_image = get_image_in_folder(path_to_Potsdam_image)
path_to_Potsdam_masks = Path("./datasets/Potsdam/labels")
paths_to_Potsdam_masks = get_image_in_folder(path_to_Potsdam_masks)

In [ ]:
paths_to_Potsdam_masks

In [ ]:
paths_to_Potsdam_image

In [ ]:
image_path

In [ ]:
image = cv2.imread(image_path)

In [ ]:
images_visualization = 1
idx = 0
random.shuffle(paths_to_Potsdam_image)
for image_path in paths_to_Potsdam_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")
    mask_path = image_path.parent.parent/'labels'/image_path.name
    mask = cv2.imread(mask_path)

    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255 

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')
    
    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

## LandCover

In [ ]:
path_to_LandCover_image = Path("./datasets/LandCover/images")
paths_to_LandCover_image = get_image_in_folder(path_to_LandCover_image)
path_to_LandCover_masks = Path("./datasets/LandCover/masks")
paths_to_LandCover_masks = get_image_in_folder(path_to_LandCover_masks)

In [ ]:
images_visualization = 1
label_to_select = 1
idx = 0
random.shuffle(paths_to_LandCover_image)
for image_path in paths_to_LandCover_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")
    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)

    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    unique_labels = np.unique(mask_np)
    print("Уникальные лейблы:", unique_labels)
    print("Количество лейблов:", len(unique_labels))
    
    binary_mask = (mask_np == label_to_select).astype(np.uint8)
    mask_np = binary_mask
    print(type(binary_mask))  
    print(binary_mask.dtype) 
    print(binary_mask.shape)
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255 

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(binary_mask, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')
    
    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
path_to_LandCover_image = Path("./datasets/LandCover/images")
paths_to_LandCover_image = get_image_in_folder(path_to_LandCover_image)
path_to_LandCover_masks = Path("./datasets/LandCover/masks")
paths_to_LandCover_masks = get_image_in_folder(path_to_LandCover_masks)

In [ ]:
px_in_patch = 512
label_to_select = 1
path_to_save_aerial = Path("./datasets/prepared_dataset/LandCover")
for i, image_path in enumerate(tqdm(paths_to_LandCover_image)):
    name = f"LandCover_{i}"
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)
    mask = (mask == label_to_select).astype(np.uint8)
    plt.imshow(binary_mask, cmap='gray')
    break
    # split_image_into_patch(image,
    #                        mask,
    #                        px_in_patch,
    #                        path_to_save_aerial,
    #                        name
    #                       )

In [ ]:
path_to_LandCover_image = Path("./datasets/prepared_dataset/LandCover/image")
paths_to_LandCover_image = get_image_in_folder(path_to_LandCover_image)
path_to_LandCover_mask = Path("./datasets/prepared_dataset/LandCover/masks")
paths_to_LandCover_mask = get_image_in_folder(path_to_LandCover_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_LandCover_image)
for image_path in paths_to_LandCover_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'masks'/image_path.name
    mask = cv2.imread(mask_path)
    
    image_np = image
    mask_np = mask.astype(np.uint8)

    if len(mask_np.shape) == 3:
        mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
    
    colored_mask = np.zeros_like(image_np)
    colored_mask[:, :, 0] = 255 

    alpha = 0.5 
    masked_image = cv2.addWeighted(image_np, 1, colored_mask, alpha, 0)

    masked_region = np.where(mask_np[:, :, None] != 0, masked_image, image_np)

    bin_ms = cv2.convertScaleAbs(mask_np)
    seg_house_im = cv2.bitwise_and(image_np, image_np, mask=bin_ms)

    plt.figure(figsize=(15, 4))

    plt.subplot(1, 4, 1)
    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.subplot(1, 4, 2)
    plt.imshow(mask_np, cmap='gray')
    plt.title('Mask')
    plt.axis('off')

    plt.subplot(1, 4, 3)
    plt.imshow(masked_region)
    plt.title('Masked Region')
    plt.axis('off')

    plt.subplot(1, 4, 4)
    plt.imshow(seg_house_im)
    plt.title('Segmented House')
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    if idx == images_visualization - 1:
        break
    idx += 1

# Реальные изображения РФ

In [ ]:
path_to_UAV_image = Path("./datasets/UAV/presentation")
paths_to_UAV_image = get_image_in_folder(path_to_UAV_image)

In [ ]:
images_visualization = 3
idx = 0
random.shuffle(paths_to_UAV_image)
for image_path in paths_to_UAV_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")

    plt.figure(figsize=(15, 4))

    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    if idx == images_visualization - 1:
        break
    idx += 1

In [ ]:
px_in_patch = 256
path_to_save_UAV = Path("./datasets/prepared_dataset/UAV")
for i, image_path in enumerate(tqdm(paths_to_UAV_image)):
    name = f"UAV_{i}"

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    split_image_into_patch(image,
                           image,
                           px_in_patch,
                           path_to_save_UAV,
                           name
                          )

In [ ]:
path_to_UAV_image = Path("./datasets/prepared_dataset/UAV/image")
paths_to_UAV_image = get_image_in_folder(path_to_UAV_image)

In [ ]:
images_visualization = 20
idx = 0
random.shuffle(paths_to_UAV_image)
for image_path in paths_to_UAV_image:

    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    print(f"image.shape = {image.shape}")

    plt.figure(figsize=(15, 4))

    plt.imshow(image)
    plt.title('Original Image')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    if idx == images_visualization - 1:
        break
    idx += 1

# Lipetsk

In [ ]:
path_to_lipetsk_image = Path("./datasets/lipetsk/images")
paths_to_lipetsk_image = get_image_in_folder(path_to_lipetsk_image)
path_to_lipetsk_mask = Path("./datasets/lipetsk/labels")
paths_to_lipetsk_mask = get_image_in_folder(path_to_lipetsk_mask)

In [ ]:
images_visualization = 10
idx = 0
random.shuffle(paths_to_lipetsk_image)
for image_path in paths_to_lipetsk_image:
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'labels'/image_path.name
    mask = cv2.imread(mask_path)

    plt.figure(figsize=(10, 4))
    
    plt.subplot(1, 2, 1)
    plt.imshow(image)
    plt.subplot(1, 2, 2)
    plt.imshow(mask)
    if idx == images_visualization:
        break
    idx += 1 

In [ ]:
path_to_lipetsk_image = Path("./datasets/lipetsk/images")
paths_to_lipetsk_image = get_image_in_folder(path_to_lipetsk_image)
path_to_lipetsk_mask = Path("./datasets/lipetsk/labels")
paths_to_lipetsk_mask = get_image_in_folder(path_to_lipetsk_mask)

In [ ]:
px_in_patch = 2000
path_to_save_lipetsk = Path("./datasets/prepared_dataset/lipetsk")
for i, image_path in enumerate(tqdm(paths_to_lipetsk_image)):
    name = f"lipetsk_{i}"
    
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    mask_path = image_path.parent.parent/'labels'/image_path.name
    mask = cv2.imread(mask_path)
    mask = cv2.cvtColor(mask, cv2.COLOR_RGB2GRAY)

    split_image_into_patch(image,
                           mask,
                           px_in_patch,
                           path_to_save_lipetsk,
                           name
                          )